In [1]:
import time
import nidaqmx
from nidaqmx.constants import AcquisitionType
from nidaqmx.errors import DaqError
import datetime as dt
import numpy as np
from pathlib import Path
import serial

In [2]:
savepath = Path("C:/Users/IPMU/Desktop/ipmu_DAQ/Pressure")
savepath.mkdir(parents=True, exist_ok=True)

filename = savepath / f"{dt.datetime.now().strftime('%Y%m%d%H%M%S')}_Pressure.csv"
print(filename)

columnname = ['Abstime', 'Reltime','Pressure1', 'Pressure2']

try:
    with open(filename, mode="a") as f:
        print(*columnname, sep=", ", file=f)
except:
    pass

C:\Users\IPMU\Desktop\ipmu_DAQ\Pressure\20260330215139_Pressure.csv


In [ ]:
t_init = time.time()
SAMPRINGRATE = 1

while True:
    
    t_abs  = dt.datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    t_rel  = time.time() - t_init
    t_arr = [t_abs, t_rel]
    
    try:
        with nidaqmx.Task() as task:
            task.ai_channels.add_ai_voltage_chan("cDAQ1Mod3/ai0")
            task.timing.cfg_samp_clk_timing(rate=SAMPRINGRATE, sample_mode=AcquisitionType.CONTINUOUS)
            data = np.array(task.read(number_of_samples_per_channel=SAMPRINGRATE))
    except DaqError as e:
        print(f"Reading Error (´；Д；｀): {e}")
        break
    
    # Pressure gauge 1
    v = float(data[0])
    p1_pa = 10 ** (2.0 * (v - 5.5))

    # Pressure gauge 2
    ser = serial.Serial('COM4', 9600,timeout=1)
    command = 'PR1' + chr(13) + chr(10);  # chr(13) = '\r'; chr(10) = '\n'
    command = command.encode('utf-8') #; print(command)   
    ser.write(command)   
    ser.readline()
    # enquiry data
    ENQ = chr(5)              # x05 in Hex, it is for data transmission
    ENQ=ENQ.encode('utf-8') 
    ser.write(ENQ) 
    p=ser.readline().decode('utf-8')
    p2_pa=p.split(',')[1]
    ser.close()

    arr = np.hstack([t_arr, p1_pa, p2_pa])

    print(arr)
    time.sleep(SAMPRINGRATE)
  
    try:
        
        with open(filename, mode="a") as f:
            print(*arr, sep=", ", file=f)

    except FileNotFoundError:

        print("File Not Found Error... (´；Д；｀)")
        filename = savepath / f"{dt.datetime.now().strftime('%Y%m%d%H%M%S')}_Pressure.csv"
        with open(filename, mode="a") as f:
            print(*columnname, sep=", ", file=f)
            print(*arr, sep=", ", file=f)